In [8]:
import pandas as pd

train_kor = pd.read_csv("../data/train_kor.csv")
train_kor.head()

,샘플 식별자 번호,고객 출생년도,고객 성별,고객명,주민번호,고객 등록일자,고객 등급,3개월 이내 금융/공동인증서 발급 여부,3개월 이내 사설인증서 발급 여부,3개월 이내 보안카드 및 OTP 발급 여부,...,마지막 ATM 거래 일자,마지막 영업점 거래 일자,7일 거래내역 중 1천만원 이상 입금 여부,7일 거래내역 중 미거래 계좌 여부,수취계좌의 거래중지계좌 해당 여부,3시간 이내 해당 수취계좌에 이체 횟수,해당 수취계좌와 거래한 횟수,60세 이후 iOS 첫 사용자,사기 시나리오 (예측 목표),계좌의 거래 재개 일자
0,TRAIN_000000,1980,male,이상호,BJWQxd-WBASPLJ,2003-01-06 18:38:01,B,0,1,0,...,2003-01-22 23:38:48,2003-01-22 23:38:48,1,1,1,0,0,0,m,2003-01-22 23:38:48
1,TRAIN_000001,1964,male,박상철,kurCwX-odPUXEt,2003-01-07 16:40:44,C,0,1,0,...,2003-01-21 21:29:08,2003-01-31 00:19:46,0,1,0,0,0,0,m,2003-01-19 21:29:08
2,TRAIN_000002,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,0,...,2003-01-31 07:13:28,2003-01-31 07:13:28,0,0,1,1,1,0,m,2003-01-31 07:13:28
3,TRAIN_000003,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,1,...,2003-01-31 11:49:56,2003-01-31 07:13:28,1,1,0,0,0,0,m,2003-01-31 07:13:28
4,TRAIN_000004,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,1,...,2003-01-31 11:49:56,2003-01-31 07:13:28,1,0,0,1,1,0,m,2003-01-31 07:13:28


In [9]:
def add_custom_features(df):
	df = df.copy()
	current_year = 2024
	auth_cols = [
            '3개월 이내 금융/공동인증서 발급 여부',
            '3개월 이내 사설인증서 발급 여부',
            '3개월 이내 보안카드 및 OTP 발급 여부',
            '3개월 이내 개인정보 수정 여부'
			]
	
	# 1. 고객 나이 및 나이 그룹
	df['고객 나이'] = current_year - df['고객 출생년도']
	df['고객 나이 그룹'] = ((df['고객 나이'] // 10) * 10).astype(str) + '대'
	
    # 2. 인증 변경 횟수
	# 인증 변경 플래그 총합: 4개의 인증 변경 여부 컬럼을 합산
	df['인증 변경 횟수'] = df[auth_cols].sum(axis=1)

    # 3. 루팅/탈옥 거래 금액
	# 루팅/탈옥 여부가 1일 때만 거래 금액 유지, 아니면 0
	df['루팅/탈옥 거래 금액'] = (
		df['이체 금액'] * (df['탈옥 및 루팅 여부'] == 1)
		)
	
	# 4. 로밍 거래 금액
	# 로밍 중 거래된 금액: 로밍 indicator가 1일 경우에만 거래 금액 반영
	df['로밍 거래 금액'] = (
		df['이체 금액'] * (df['모바일 로밍 여부'] == 1)
		)
	
	# 5. 잔액 변동
	df['잔액 변동'] = (
    df['거래 후 잔액'] - df['거래 전 잔액']
	)

	# 6. 일일 한도 사용 금액
	df['일일 한도 사용 금액'] = (
    df['1일 거래 한도'] -df['1일 거래 한도 잔여액']
	)

	# 7. 일일 한도 대비 거래 비율
	df['일일 한도 대비 거래 비율'] = df['이체 금액'] / (df['1일 거래 한도'] + 1e-6)

	# 8. 거래 금액 상한선
	# 정규분포 가정 하에 통상적인 최대 거래 범위를 추정하고자 (이체 금액 + 표준편차)로 나타냄
	# 근데 이체 금액 중 0 미만인 값들도 존재하니 일단 아래처럼 코드처럼 제작
	df['거래 금액 상한선'] = df['이체 금액'] + df['1개월 거래내역 이체(출금) 금액 표준편차(중앙값)']

	# 9. 상한선 초과 여부
	df['상한선 초과 여부'] = (
		df['1개월 거래내역 중 최대 이체(출금) 금액'] > df['거래 금액 상한선']
		).astype(int)
	
	# 10. 계좌 생성 - 고객 등록 시간 차
	# datetime 형 변환 (한 번만 실행하면 됨)
	df['계좌 개설 일자'] = pd.to_datetime(df['계좌 개설 일자'], errors='coerce')
	df['고객 등록일자'] = pd.to_datetime(df['고객 등록일자'], errors='coerce')
	# 시간 차이 계산 (초 단위)
	df['계좌 생성_고객 등록 시간 차'] = (
		df['계좌 개설 일자'] - df['고객 등록일자']
		).dt.total_seconds()

	# 11. 계좌 생성 - 거래 발생 시간 차
	# datetime 변환 (이미 되어있으면 생략 가능)
	df['거래일자'] = pd.to_datetime(df['거래일자'], errors='coerce')
	
	df['계좌 생성_거래 발생 시간 차'] = (
	df['거래일자'] - df['계좌 개설 일자']
	).dt.total_seconds()
	
	# 12. 마지막 ATM 거래 후 경과 시간
	# datetime 변환 (이미 되어 있으면 생략 가능)
	df['마지막 ATM 거래 일자'] = pd.to_datetime(df['마지막 ATM 거래 일자'], errors='coerce')

	# 경과 시간 계산
	df['마지막 ATM 거래 후 경과 시간'] = (
		df['거래일자'] - df['마지막 ATM 거래 일자']
		).dt.total_seconds()

	# 13. 마지막 지점 거래 후 경과 시간
	# datetime 변환
	df['마지막 영업점 거래 일자'] = pd.to_datetime(
		df['마지막 영업점 거래 일자'], 
		errors='coerce')

	# 경과 시간 계산
	df['마지막 지점 거래 후 경과 시간'] = (
		df['거래일자'] - df['마지막 영업점 거래 일자']
		).dt.total_seconds()

	# 14. 고래 재개 후 경과 시간
	# datetime 변환
	df['계좌의 거래 재개 일자'] = pd.to_datetime(
		df['계좌의 거래 재개 일자'], 
		errors='coerce')

	# 경과 시간 계산
	df['거래 재개 후 경과 시간'] = (
		df['거래일자'] - df['계좌의 거래 재개 일자']
		).dt.total_seconds()
	
	return df

In [13]:
add_custom_features(train_kor).head()

,샘플 식별자 번호,고객 출생년도,고객 성별,고객명,주민번호,고객 등록일자,고객 등급,3개월 이내 금융/공동인증서 발급 여부,3개월 이내 사설인증서 발급 여부,3개월 이내 보안카드 및 OTP 발급 여부,...,잔액 변동,일일 한도 사용 금액,일일 한도 대비 거래 비율,거래 금액 상한선,상한선 초과 여부,계좌 생성_고객 등록 시간 차,계좌 생성_거래 발생 시간 차,마지막 ATM 거래 후 경과 시간,마지막 지점 거래 후 경과 시간,거래 재개 후 경과 시간
0,TRAIN_000000,1980,male,이상호,BJWQxd-WBASPLJ,2003-01-06 18:38:01,B,0,1,0,...,880000,0,0.0050,10000,0,1400447.0,254506.0,254506.0,254506.0,254506.0
1,TRAIN_000001,1964,male,박상철,kurCwX-odPUXEt,2003-01-07 16:40:44,C,0,1,0,...,-1266850,0,-25.1600,-12382581,1,1054104.0,968014.0,795214.0,7376.0,968014.0
2,TRAIN_000002,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,0,...,21698509,49000000,0.4026,20130000,0,1703092.0,11204.0,11204.0,11204.0,11204.0
3,TRAIN_000003,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,1,...,26188509,1000000,12.3100,25341248,0,1703092.0,21633.0,5045.0,21633.0,21633.0
4,TRAIN_000004,1982,female,조옥자,OiERQa-CTXBoaX,2003-01-11 14:08:36,B,1,1,1,...,-16130000,1030000,-0.0150,1091487,1,1703092.0,29338.0,12750.0,29338.0,29338.0
